# Ray recovery experiments on GCP

**Project:** `research-488322` · **Image:** `ray-built-v1` · **Zone:** `us-central1-a`

This notebook creates **five Standard e2-standard-4 VMs (4 vCPUs each)**, runs matched CPU recovery experiments, downloads reports and PNG/PDF plots, and includes optional deletion at the end. It uses the Ray build already stored in your image; it does not rebuild Ray or change that image.

**Run cells in order.** Run this notebook outside the disposable experiment VMs. Five VMs remain billable until you execute cleanup, even if the notebook disconnects or a test fails.

Two comparisons are included:

| Comparison | What it measures |
|---|---|
| Ordinary Ray DDP checkpoint retry vs TorchFT + aligned Ray Data | Combined integration benefit and healthy overhead |
| TorchFT + indexed input vs TorchFT + retained aligned input | Input strategy contribution with the same peer recovery |

This suite uses synthetic CPU training. It does **not** enable Fixed-R. The first comparison includes TorchFT's existing recovery capabilities. Negative results and failed correctness checks remain visible.

**Failure scope:** SIGKILL of the entire Ray worker service on one VM, including its actors and object store. The VM itself stays running. The worker service stays down for the rest of that observation. Head and shared checkpoint storage survive.

**Validation status:** source reviewed, Python compilation and offline orchestration tests passed; no live GCP execution was performed while preparing this notebook. Start with the smoke tests.


## 1. Authenticate the notebook host

Requirements: Python 3.10+, Jupyter/IPython, [Google Cloud CLI](https://cloud.google.com/sdk/docs/install), and OpenSSH (`ssh`, `ssh-keygen`). Ray/PyTorch are needed on the saved image, not on this notebook host.

In a terminal **on the notebook host**, authenticate once if needed:

```bash
gcloud auth login --no-launch-browser
gcloud config set project research-488322
```

The account needs Compute resource management, VM metadata/SSH access, and **IAP-secured Tunnel User** access. This notebook uses an isolated VPC and IAP SSH; it never opens Ray ports to the public internet. The VMs use external IPs for package downloads. Metadata-based SSH is requested; an organization enforcing OS Login needs an adapted setup.

The next cell checks the active login and enables the Compute/IAP APIs. It creates no VMs.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

for tool in ("gcloud", "ssh", "ssh-keygen"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"Install {tool} on the notebook host and restart Jupyter if PATH changed.")

def cli(*args):
    p = subprocess.run(["gcloud", *args], text=True, capture_output=True)
    if p.returncode:
        raise RuntimeError(p.stderr)
    return p.stdout

print(cli("auth", "list", "--filter=status:ACTIVE", "--format=table(account,status)"))
print(cli("services", "enable", "compute.googleapis.com", "iap.googleapis.com",
          "--project=research-488322", "--quiet"))


## 2. Extract the included Python code

This notebook is self-contained. The collapsed cell carries the controller, physical-cluster backend, matched benchmark, plotter, and README. It verifies the embedded archive and extracts readable `.py` files beside the notebook. You can inspect or copy those files before provisioning.


In [ ]:
import base64
import hashlib
import io
import sys
import zipfile
from pathlib import Path

BUNDLE_SHA256 = "760d7b3df27197419582ffea39133b64a3a9c44f4348544bb0aed6e22f57532e"
BUNDLE_BASE64 = "..."
blob = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(blob).hexdigest() == BUNDLE_SHA256
BUNDLE_DIR = Path("ray_gcp_harness_760d7b3df2").resolve()
BUNDLE_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for member in archive.infolist():
        target = (BUNDLE_DIR / member.filename).resolve()
        if not target.is_relative_to(BUNDLE_DIR):
            raise ValueError("Unexpected archive path")
        target.parent.mkdir(parents=True, exist_ok=True)
        data = archive.read(member)
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError(f"Local edits found in {target}; preserve them before extracting again.")
        target.write_bytes(data)
sys.path.insert(0, str(BUNDLE_DIR))
from gcp_launcher import GCPExperiment
print("Python source files:", BUNDLE_DIR)


## 3. Configure and create the experiment

Leave `STATE_FILE = None` for a **new** experiment. After a kernel restart, set it to the printed `*-state.json` file to reconnect to the existing experiment. Do not create a second cluster to resume a first one.

The state file records exact VM names, cloud configuration, suite status and downloaded results. Keep it until cleanup completes.


In [ ]:
PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
WORKDIR = "ray_gcp_results"
STATE_FILE = None  # To resume: "ray_gcp_results/rayexp-...-state.json"

exp = GCPExperiment(project=PROJECT, zone=ZONE, image=IMAGE,
                    workdir=WORKDIR, state_file=STATE_FILE)
print("Planned instances:", *exp.names, sep="\n  ")


**The next cell starts billable resources.** It creates the five VMs, installs NFS packages, sets up shared checkpoint storage on the head, and checks imports and the pinned source revision. All nodes use the same image and software paths.

If setup fails after creation, inspect the error. `exp.setup()` retries setup on the same VMs; do not rerun provisioning. The cleanup cell also works after partial provisioning with the explicit discard option if necessary.


In [ ]:
if STATE_FILE is None:
    exp.provision()
else:
    print("Loaded existing experiment; provisioning skipped.")
    print("Ready:", exp.state.get("ready", False))
    # If the prior attempt stopped during setup, run: exp.setup()


## 4. Smoke tests — run before the full sweep

Each smoke comparison has four observations: both methods, healthy and failed. Both use 60 steps, checkpoint interval 30, failure before update 49, batch 32 and width 512. Each observation gets a fresh Ray cluster.

A passing suite requires model agreement, correct checkpoint restoration and rollback counts, the expected worker restarts, and training ranks on different worker VMs. The driver runs on the head, so interrupting this notebook only stops monitoring; use the interruption controls below to stop the remote work.


In [ ]:
def run_or_resume(name, **kwargs):
    previous = exp.state["suites"].get(name)
    if previous is None:
        return exp.run_suite(name, **kwargs)
    if previous.get("state") == "finished" and previous.get("exit_code") == 0:
        path = Path(previous.get("downloaded", ""))
        return path if (path / "report.json").exists() else exp.download(name)
    if previous.get("state") in ("running", "launching"):
        return exp.wait_suite(name)
    raise RuntimeError(f"{name} previously failed/interrupted. Inspect evidence before choosing a fresh suite name.")

smoke_options = dict(repeats=1, steps=60, batch_size=32, model_width=512,
                     checkpoint_every=30, fault_step=49, timeout_s=1200)
smoke_ordinary = run_or_resume("smoke-ordinary", comparison="ordinary-ray", **smoke_options)
smoke_input = run_or_resume("smoke-input", comparison="torchft-input", **smoke_options)
print("Both smoke suites passed.")


In [ ]:
from IPython.display import display, Image, Markdown

def show_suite(name):
    folder = Path(exp.state["suites"][name]["downloaded"])
    report = json.loads((folder / "report.json").read_text())
    samples = report["samples"]
    valid = sum(s.get("matches_reference") is True and s.get("recovery_contract_valid") is True
                for s in samples)
    display(Markdown(f"### {name}: {valid}/{len(samples)} validated observations"))
    if (folder / "report.png").exists():
        display(Image(filename=str(folder / "report.png")))
    print("JSON, PNG/PDF, checkpoints and logs:", folder)
    return report

show_suite("smoke-ordinary")
show_suite("smoke-input");


## 5. Full sweep — enable after both smoke tests pass

Set `RUN_FULL_SUITE = True`. Four suites run sequentially on the **same five VMs**. Each suite uses three repetitions (12 observations), for 48 observations total. The runner alternates method order by repetition. This can take hours; completion time depends on recovery delays and your VMs.

All runs use 300 steps, failure before update 239, batch 256, width 2048 and no artificial startup delay. The full sweep stops if a suite has invalid/failed observations. It does not retry failures until they look successful.


In [ ]:
RUN_FULL_SUITE = False

if RUN_FULL_SUITE:
    for smoke in ("smoke-ordinary", "smoke-input"):
        state = exp.state["suites"].get(smoke, {})
        if state.get("state") != "finished" or state.get("exit_code") != 0:
            raise RuntimeError("Both smoke suites must pass before the full sweep.")
    for interval in (30, 60, 120):
        run_or_resume(f"ordinary-cp{interval}", comparison="ordinary-ray",
                      repeats=3, steps=300, batch_size=256, model_width=2048,
                      checkpoint_every=interval, fault_step=239, timeout_s=1800)
    run_or_resume("input-cp120", comparison="torchft-input",
                  repeats=3, steps=300, batch_size=256, model_width=2048,
                  checkpoint_every=120, fault_step=239, timeout_s=1800)
else:
    print("Full sweep skipped. Set RUN_FULL_SUITE = True when ready.")


## 6. Display results

Every figure includes healthy runtime, runtime under worker service loss, recovery to new progress, and repeated committed updates. Bars show medians with individual-run points and validated/total counts. Only complete valid pairs support the paired percentage changes.

A **positive failure-runtime reduction** means faster recovery runs; a **positive healthy overhead** means slower healthy runs. Retain negative results. The ordinary-Ray comparison is a combined-system comparison, not evidence that input retention alone caused a speedup.


In [ ]:
import statistics

rows = ["| Suite | Valid pairs | Median failure-runtime reduction | Median healthy overhead |",
        "|---|---:|---:|---:|"]
for name in ("ordinary-cp30", "ordinary-cp60", "ordinary-cp120", "input-cp120"):
    state = exp.state["suites"].get(name, {})
    folder = Path(state.get("downloaded", "/nonexistent"))
    if not (folder / "report.json").exists():
        continue
    report = show_suite(name)
    pairs = [p for p in report.get("pairs", []) if p.get("valid")]
    if pairs:
        reduction = statistics.median(p["fault_runtime_reduction_pct"] for p in pairs)
        overhead = statistics.median(p["healthy_fit_overhead_pct"] for p in pairs)
        rows.append(f"| {name} | {len(pairs)}/{report['parameters']['repeats']} | {reduction:+.1f}% | {overhead:+.1f}% |")
    else:
        rows.append(f"| {name} | 0 | No valid comparison | No valid comparison |")
display(Markdown("\n".join(rows)))


## 7. Reconnect, stop, or collect partial results

If the notebook disconnects, the current suite continues on the head with a bounded service lifetime. The VMs remain running afterward. Reload this notebook's extraction/configuration cells using the saved `STATE_FILE`, then use the appropriate command below.

Do not call `stop_services()` while you want the suite to continue. It stops only this experiment's jobs; it does not delete VMs.


In [ ]:
# Uncomment only the operation you need:
# exp.wait_suite("ordinary-cp120")   # Continue monitoring an already launched suite.
# exp.stop_services()               # Stop remote work but retain VMs and results.
# exp.download("ordinary-cp120")     # Download completed or partial evidence.

print("State file:", exp.state_path)
print("Recorded suites:", {n: s.get("state") for n, s in exp.state["suites"].items()})


## 8. Optional deletion — all instances created by this notebook

Set `DELETE_INSTANCES = True` to delete this experiment's **five VMs and their boot disks**, plus its firewall rules, subnet and VPC. The launcher checks the recorded names and experiment labels. It never deletes `ray-built-v1`, your project, or unrelated instances.

Collect results first. If setup failed or you deliberately want to discard remote evidence, also set `DISCARD_UNCOLLECTED = True`. This override permanently discards uncollected results on those VM disks. The local downloads remain.


In [ ]:
DELETE_INSTANCES = False
DISCARD_UNCOLLECTED = False

if DELETE_INSTANCES:
    exp.cleanup(confirm=exp.run_id, allow_uncollected=DISCARD_UNCOLLECTED)
else:
    print("Cleanup skipped. Experiment VMs remain billable.")
    print("Saved image is retained either way:", exp.state["image"])


## Scope and provenance

The bundled README documents the failure model, timing semantics, NFS storage, permissions and recovery checks. Read it with:

```python
print((BUNDLE_DIR / "README.md").read_text())
```

The benchmark core comes from your fork at `dc4703753c8f3d52e74c389c72449dacc27b41f3`. Its model/recovery checks are retained. GCP transport and an explicit private head-IP Lighthouse binding are added. Source, package and native-extension identities are recorded on the VMs.

Reference documentation: [GCP custom images](https://cloud.google.com/compute/docs/instances/create-vm-from-custom-image), [IAP TCP forwarding](https://cloud.google.com/iap/docs/using-tcp-forwarding), [instance deletion](https://cloud.google.com/sdk/gcloud/reference/compute/instances/delete).
